<a href="https://colab.research.google.com/github/davidlealo/practicos_sisrec_2026/blob/main/practico04/04_content_based_imagenes_respuestaDLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctico Content-based (Imágenes)

IIC-3633, PUC Chile

**Profesor:** Denis Parra

**Alumno:** `David Leal`

En esta actividad trabajaremos con un recomendador de ropa basado netamente en las imagenes más similares extrayendo features con redes neuronales convolucionales.



In [ ]:
from keras.applications import vgg16, vgg19, ResNet50
from tensorflow.keras.utils import load_img,img_to_array
from keras.models import Model
from keras.applications.imagenet_utils import preprocess_input

from PIL import Image
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

En esta sección se trabajará con modelos pre-entrenados de redes convolucionales (CNN) que extraen caracteristicas visuales de las imagenes.

![Ejemplo de red convolucional](https://www.researchgate.net/publication/326658868/figure/fig3/AS:962202072805390@1606418259908/AlexNet-architecture-This-shows-the-process-to-obtain-the-latent-feature-vector-we-use.png)


Para los curiosos se recomienda revisar los siguientes links:

- Artículo: [Understand Deep Residual Networks](https://medium.com/@14prakash/understanding-and-implementing-architectures-of-resnet-and-resnext-for-state-of-the-art-image-cf51669e1624)
- [Keras applications](https://keras.io/applications/)

# Descarga de imagenes

In [ ]:
%%capture
!gdown 1iLOeNZw69iyYXa7QS5ZutN7ACUpkbL3x
!unzip images_fashion.zip
!mkdir images
!mv *.png images/

# Cargamos la CNN pre-entrenada en ImageNet


In [ ]:
# cargamos el modelo escoger

modelo_escogido = 'vgg16' #@param["vgg16", "vgg19"]

if modelo_escogido == 'vgg16':
  # cargar modelo
  vgg_model = vgg16.VGG16(weights='imagenet')
  # quitar la capa de clasificacion
  feat_extractor = Model(inputs=vgg_model.input, outputs=vgg_model.get_layer("fc2").output)
  # vemos resumen de la arquitectura del modelo
  feat_extractor.summary()

elif modelo_escogido == 'vgg19':
  # cargar modelo
  vgg19_model = vgg19.VGG19(weights='imagenet')
  # quitar la capa de clasificacion
  feat_extractor = Model(inputs=vgg19_model.input, outputs=vgg19_model.get_layer("fc2").output)
  # vemos resumen de la arquitectura del modelo
  feat_extractor.summary()



## Procesamiento de imágenes para dárselas como input a la CNN

In [ ]:
ls

In [ ]:
imgs_path = "images/" # ruta

imgs_model_width, imgs_model_height = 224, 224 # tamaño de las imagenes 224x224 pixeles

nb_closest_images = 5 # cantidad de imagenes similares a recomendar

In [ ]:
files = [imgs_path + x for x in os.listdir(imgs_path) if "png" in x]
print("total de imagenes:",len(files))

In [ ]:
# vemos imagen aleatoria
import random

idx =  random.randint(0, len(files))
original = load_img(files[idx], target_size=(imgs_model_width, imgs_model_height))
plt.imshow(original)
plt.show()
print("image cargada exitosamente!")

In [ ]:
# convertir PIL image a numpy array
numpy_image = img_to_array(original)

# convertir imagen a batch de imagenes para entrenamiento más eficiente
image_batch = np.expand_dims(numpy_image, axis=0)
print('image batch size', image_batch.shape)

# preparamos la imagen para la VGG16
processed_image = preprocess_input(image_batch.copy())

processed_image

In [ ]:
# obtenemos los features (embeddings) de las imagenes pasandolas por la VGG16
img_features = feat_extractor.predict(processed_image)

print("features successfully extracted!")
print("number of image features:",img_features.size)
img_features

In [ ]:
img_features.shape

In [ ]:
# repetimos el mismo proceso para todas las imagenes y guardamos los batch en una lista para entregarselos procesados a la VGG16
importedImages = []

for f in files:
    filename = f
    original = load_img(filename, target_size=(224, 224))
    numpy_image = img_to_array(original)
    image_batch = np.expand_dims(numpy_image, axis=0)

    importedImages.append(image_batch)

images = np.vstack(importedImages)

processed_imgs = preprocess_input(images.copy())

In [ ]:
# obtenemos los features para cada imagen con la CNN
imgs_features = feat_extractor.predict(processed_imgs)

print("features extraidos exitosamente!")
imgs_features.shape

In [ ]:
# computa similaridad coseno entre los features de las imagenes
cosSimilarities = cosine_similarity(imgs_features)

# guardamos los resultados en un dataframe
cos_similarities_df = pd.DataFrame(cosSimilarities, columns=files, index=files)
cos_similarities_df #.head()

In [ ]:
# esta funcion recupera las imagenes más similares dada una imagen entregada por el usuario
def retrieve_most_similar_products(given_img):

    print("-----------------------------------------------------------------------")
    print("producto escogido:")

    original = load_img(given_img, target_size=(imgs_model_width, imgs_model_height))
    plt.imshow(original)
    plt.show()

    print("-----------------------------------------------------------------------")
    print("productos más similares:")

    closest_imgs = cos_similarities_df[given_img].sort_values(ascending=False)[1:nb_closest_images+1].index
    closest_imgs_scores = cos_similarities_df[given_img].sort_values(ascending=False)[1:nb_closest_images+1]

    for i in range(0,len(closest_imgs)):
        original = load_img(closest_imgs[i], target_size=(imgs_model_width, imgs_model_height))
        plt.imshow(original)
        plt.show()
        print("score de similaridad : ",closest_imgs_scores[i])

In [ ]:
idx = 1100 # random.randint(0, len(files))
print(idx)
print(files[idx])
retrieve_most_similar_products(files[idx])

# ACTIVIDAD

1. Mostrar 2 ejemplos de búsqueda de imagenes similares utilizando ambas arquitecturas (VGG16 y VGG19) e imprimir los resultados. (3 ptos)

2. ¿Cuál de las dos arquitecturas (VGG16 o VGG19) tiene más parámetros entrenables después de quitar la última capa de clasificación?. Justifique indicando la cantidad de parámetros de cada una. (2 ptos)


Pregunta 1:

# Dos búsquedas VGG16

In [ ]:
# busqueda 1
idx = 1 # VGG16
print(idx)
print(files[idx])
retrieve_most_similar_products(files[idx])

In [ ]:
# busqueda 2
idx = 2 # VGG16
print(idx)
print(files[idx])
retrieve_most_similar_products(files[idx])

VGG 19

In [ ]:
# cargamos el modelo escoger

modelo_escogido = 'vgg19' #@param["vgg16", "vgg19"]

if modelo_escogido == 'vgg16':
  # cargar modelo
  vgg_model = vgg16.VGG16(weights='imagenet')
  # quitar la capa de clasificacion
  feat_extractor = Model(inputs=vgg_model.input, outputs=vgg_model.get_layer("fc2").output)
  # vemos resumen de la arquitectura del modelo
  feat_extractor.summary()

elif modelo_escogido == 'vgg19':
  # cargar modelo
  vgg19_model = vgg19.VGG19(weights='imagenet')
  # quitar la capa de clasificacion
  feat_extractor = Model(inputs=vgg19_model.input, outputs=vgg19_model.get_layer("fc2").output)
  # vemos resumen de la arquitectura del modelo
  feat_extractor.summary()

In [ ]:
ls

In [ ]:
imgs_path = "images/" # ruta

imgs_model_width, imgs_model_height = 224, 224 # tamaño de las imagenes 224x224 pixeles

nb_closest_images = 5 # cantidad de imagenes similares a recomendar


In [ ]:
files = [imgs_path + x for x in os.listdir(imgs_path) if "png" in x]
print("total de imagenes:",len(files))

In [ ]:
# vemos imagen aleatoria
import random

idx =  random.randint(0, len(files))
original = load_img(files[idx], target_size=(imgs_model_width, imgs_model_height))
plt.imshow(original)
plt.show()
print("image cargada exitosamente!")

In [ ]:
# convertir PIL image a numpy array
numpy_image = img_to_array(original)

# convertir imagen a batch de imagenes para entrenamiento más eficiente
image_batch = np.expand_dims(numpy_image, axis=0)
print('image batch size', image_batch.shape)

# preparamos la imagen para la VGG16
processed_image = preprocess_input(image_batch.copy())

processed_image

In [ ]:
# obtenemos los features (embeddings) de las imagenes pasandolas por la VGG16
img_features = feat_extractor.predict(processed_image)

print("features successfully extracted!")
print("number of image features:",img_features.size)
img_features

In [ ]:
img_features.shape

In [ ]:
# repetimos el mismo proceso para todas las imagenes y guardamos los batch en una lista para entregarselos procesados a la VGG16
importedImages = []

for f in files:
    filename = f
    original = load_img(filename, target_size=(224, 224))
    numpy_image = img_to_array(original)
    image_batch = np.expand_dims(numpy_image, axis=0)

    importedImages.append(image_batch)

images = np.vstack(importedImages)

processed_imgs = preprocess_input(images.copy())

In [ ]:
# obtenemos los features para cada imagen con la CNN
imgs_features = feat_extractor.predict(processed_imgs)

print("features extraidos exitosamente!")
imgs_features.shape

In [ ]:
# computa similaridad coseno entre los features de las imagenes
cosSimilarities = cosine_similarity(imgs_features)

# guardamos los resultados en un dataframe
cos_similarities_df = pd.DataFrame(cosSimilarities, columns=files, index=files)
cos_similarities_df #.head()

In [ ]:
# esta funcion recupera las imagenes más similares dada una imagen entregada por el usuario
def retrieve_most_similar_products(given_img):

    print("-----------------------------------------------------------------------")
    print("producto escogido:")

    original = load_img(given_img, target_size=(imgs_model_width, imgs_model_height))
    plt.imshow(original)
    plt.show()

    print("-----------------------------------------------------------------------")
    print("productos más similares:")

    closest_imgs = cos_similarities_df[given_img].sort_values(ascending=False)[1:nb_closest_images+1].index
    closest_imgs_scores = cos_similarities_df[given_img].sort_values(ascending=False)[1:nb_closest_images+1]

    for i in range(0,len(closest_imgs)):
        original = load_img(closest_imgs[i], target_size=(imgs_model_width, imgs_model_height))
        plt.imshow(original)
        plt.show()
        print("score de similaridad : ",closest_imgs_scores[i])

# Dos búsquedas VGG19

In [ ]:
# busqueda 1
idx = 1 # VGG19
print(idx)
print(files[idx])
retrieve_most_similar_products(files[idx])

In [ ]:
# busqueda 2
idx = 2 # VGG19
print(idx)
print(files[idx])
retrieve_most_similar_products(files[idx])

Pregunta 2
- parámetros entrenables VGG16 sin última capa
- parámetros entrenables VGG19 sin última capa




In [ ]:
vgg16_input_layer = 0
vgg16_block_1 = 1792 + 36928 +0
vgg16_block_2 = 73856 + 147584 + 0
vgg16_block_3 = 295168 + (590080 * 2)
vgg16_block_4 = 1180160 + (2359808 * 2) + 0
vgg16_block_5 =  2359808 * 3 + 0
vgg16_total = 1180160 + (2359808 * 3) + 0
vgg16_flaten_layer = 0
vgg16_fc1 = 102764544
vgg16_fc2 = 16781312
total_params_vgg16 = 134260544

In [ ]:
# Contar parámetros de vgg16
suma_vgg16 = vgg16_input_layer + vgg16_block_1 + vgg16_block_2 + vgg16_block_3 + vgg16_block_4 + vgg16_block_5 + vgg16_flaten_layer + vgg16_fc1 + vgg16_fc2
print(suma_vgg16)

In [ ]:
suma_vgg16_without_last_layer = vgg16_input_layer + vgg16_block_1 + vgg16_block_2 + vgg16_block_3 + vgg16_block_4 + vgg16_block_5 + vgg16_flaten_layer + vgg16_fc1
print(suma_vgg16_without_last_layer)

In [ ]:
vgg19_input_layer = 0
vgg19_block_1 = 1792 + 36928 +0
vgg19_block_2 = 73856 + 147584 + 0
vgg19_block_3 = 295168 + (590080 * 3)
vgg19_block_4 = 1180160 + (2359808 * 3) + 0
vgg19_block_5 =  2359808 * 4 + 0
vgg19_total = 1180160 + (2359808 * 3) + 0
vgg19_flaten_layer = 0
vgg19_fc1 = 102764544
vgg19_fc2 = 16781312
total_params_vgg19 = 139570240

In [ ]:
# Contar parámetros de vgg19
suma_vgg19 = vgg19_input_layer + vgg19_block_1 + vgg19_block_2 + vgg19_block_3 + vgg19_block_4 + vgg19_block_5 + vgg19_flaten_layer + vgg19_fc1 + vgg19_fc2
print(suma_vgg19)

In [ ]:
suma_vgg19_without_last_layer = vgg19_input_layer + vgg19_block_1 + vgg19_block_2 + vgg19_block_3 + vgg19_block_4 + vgg19_block_5 + vgg19_flaten_layer + vgg19_fc1
print(suma_vgg19_without_last_layer)

# RESPUESTA: Pregunta 2

Los parámetros entrenables de VGG16 son 134260544 pero eliminando la última capa entrenable es de 117479232

Los parámetros entrenables de VGG19 son 139570240 pero eliminando la última capa entrenable es de 122788928

# Uso de IA

Usé IA para comprender el tema de las capas, igual fue porque me dio inseguridad, entonces le pedí en un prompt inicial que no me diera respuestas, sino que me explicara lo que le estaba preguntando y nada más y al final le pregunté si las capas las sumaba o las multiplicaba. Me preocupé de no pedir directamente la respuesta, aunque me apoyé mucho. Tal vez como autocrítica si yo hubiera realizado solo las operaciones me hubiera dado cuenta que las sumaba y no multiplicaba, incluso al ojo uno se daba cuenta de eso. De todas formas acá la conversación: https://gemini.google.com/share/e361aba58745

También lo dejé en github https://github.com/davidlealo/practicos_sisrec_2026/blob/main/practico04/04_content_based_imagenes_respuestaDLO.ipynb